<a href="https://colab.research.google.com/github/TheOohWee/CSE-10124-LAB-3/blob/main/Amir_CSE10124_lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE 10124 - Building ChatGPT: Lab 03 (5 pts.)

- NETID:

## Overview

In Labs 01 and 02, we built the low-level building blocks of a neural network: embeddings, linear layers, and activation functions. In this lab, we assemble those pieces into the **Transformer Block** — the core repeating unit inside models like GPT, BERT, and ChatGPT.

By the end of this lab, you will have a working, from-scratch implementation of a single Transformer block that you could stack $N$ times to build a full language model.

### What We're Building

A Transformer block takes an input $X$ of shape $(B, T, C)$ and produces an output $Y$ of the same shape. Internally, the data flows through two sub-blocks connected by residual additions:

1. The input is first normalized by **Layer Norm 1**, then passed through **Self-Attention**, and the result is added back to the original input (residual connection).
2. That sum is then normalized by **Layer Norm 2**, passed through a **Feed-Forward Network**, and again added back via a residual connection to produce the final output.

The four main components we will implement:

1. **Layer Normalization** — stabilizes training by normalizing activations
2. **Scaled Dot-Product Self-Attention** — lets each token "look at" other tokens to gather context
3. **Feed-Forward Network (FFN)** — processes each token independently through a small neural network
4. **Residual Connections** — "highways" that let gradients flow easily through deep networks

![Transformer Block Internals](https://jalammar.github.io/images/gpt2/gpt2-self-attention-example-2.png)

*Image: Inside a single Transformer decoder block — the input flows through Masked Self-Attention (with attention weights shown), then through the Feed-Forward Neural Network. (Source: [Jay Alammar — The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/))*

### Topics Covered

| Task ID | Description | Points |
|---------|-------------|--------|
| 00 | Data + Library Loading | 0 |
| 01 | Layer Normalization | 1 |
| &nbsp;&nbsp;&nbsp;&nbsp;01-1 | &nbsp;&nbsp;&nbsp;&nbsp;- LayerNorm Class | |
| &nbsp;&nbsp;&nbsp;&nbsp;01-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Comparison to `nn.LayerNorm` | |
| 02 | Scaled Dot-Product Self-Attention | 2 |
| &nbsp;&nbsp;&nbsp;&nbsp;02-1 | &nbsp;&nbsp;&nbsp;&nbsp;- `scaled_dot_product_attention` Function | |
| &nbsp;&nbsp;&nbsp;&nbsp;02-2 | &nbsp;&nbsp;&nbsp;&nbsp;- `SingleHeadAttention` Class | |
| &nbsp;&nbsp;&nbsp;&nbsp;02-3 | &nbsp;&nbsp;&nbsp;&nbsp;- Comparison to `F.scaled_dot_product_attention` | |
| 03 | Feed-Forward Sub-Block | 1 |
| &nbsp;&nbsp;&nbsp;&nbsp;03-1 | &nbsp;&nbsp;&nbsp;&nbsp;- `FFN` Class | |
| &nbsp;&nbsp;&nbsp;&nbsp;03-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Short Answer Questions | |
| 04 | Transformer Block | 1 |
| &nbsp;&nbsp;&nbsp;&nbsp;04-1 | &nbsp;&nbsp;&nbsp;&nbsp;- `TransformerBlock` Class | |
| &nbsp;&nbsp;&nbsp;&nbsp;04-2 | &nbsp;&nbsp;&nbsp;&nbsp;- Causal Masking Demo | |
| 05 | Generate Submission | 0 |

Please complete all sections. Some questions will require written answers, while others will involve coding. Be sure to run your code cells to verify your solutions.

In the code blocks, look for comments like:
```python
# TODO: some task description
```
as these are what you will be expected to fill in. For code sections longer than a line, there's an additional comment:
```python
# LINES: N
```
where `N` is the number of lines I have in my solution. This is NOT graded on, it is just a reference for you — if you have way more or way fewer it may be worth rethinking your code, but +/- 2-3 lines is nothing to worry about usually.

## Task 00: Setup (0 pts.)
#### Loading Data and Importing Libraries

Run the cell below to download the course repository and import the building blocks we wrote in previous labs (`LinearLayer`, `ReLU`). These will be used later when we assemble the full Transformer block.

### Task 00: Code (0 pts.)


In [1]:
import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

# Import the building blocks from previous labs:
# - uts: utility functions (tokenizer helpers, etc.)
# - LinearLayer: our from-scratch linear transformation (X @ W^T + b)
# - ReLU: our from-scratch activation function (max(0, x))
import irishGPT.utilities as uts
from irishGPT.linear_layer import LinearLayer
from irishGPT.relu import ReLU


/content
rm: cannot remove '/content/nd-cse-10124-lectures': No such file or directory
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 466 (delta 19), reused 41 (delta 11), pack-reused 411 (from 1)
Receiving objects: 100% (466/466), 60.58 MiB | 18.63 MiB/s, done.
Resolving deltas: 100% (282/282), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures


---

## Task 01: Layer Normalization

### Why Do We Need Normalization?

When training deep neural networks, the values flowing through the network can grow very large or very small as they pass through successive layers. This causes two problems:

1. **Unstable gradients** — if activations are huge, gradients explode; if tiny, gradients vanish
2. **Slow convergence** — the optimizer has to deal with wildly different scales across features

**Layer Normalization** solves this by re-centering and re-scaling the activations at each layer. For each token independently, it:
1. Computes the mean and standard deviation across the embedding dimension
2. Normalizes to zero mean and unit variance
3. Applies a learned scale ($\gamma$) and shift ($\beta$) so the model can undo the normalization if needed

### The Math

Given an input vector $\mathbf{x}$ (one token's embedding of dimension $C$):

$$\mu = \frac{1}{C} \sum_{i=1}^{C} x_i \qquad \sigma = \sqrt{\frac{1}{C} \sum_{i=1}^{C} (x_i - \mu)^2 + \epsilon}$$

$$\hat{x}_i = \frac{x_i - \mu}{\sigma} \qquad y_i = \gamma_i \cdot \hat{x}_i + \beta_i$$

- $\epsilon$ is a tiny constant (e.g., $10^{-5}$) to avoid division by zero
- $\gamma$ and $\beta$ are **learnable** parameters (initialized to 1 and 0)
- The normalization happens **per token** — each token in the sequence is normalized independently

### Visual: What Layer Norm Does

Imagine a batch of 2 sequences, each with 3 tokens, each token having a 4-dimensional embedding. The input tensor $X$ has shape $(B{=}2, T{=}3, C{=}4)$. LayerNorm normalizes **across the embedding dimension** (the last axis, $C$) for each token independently. So each row (token) is normalized to approximately zero mean and unit variance, regardless of what other tokens or batches look like.



### Task 01-1: Description (0 pts.)
#### LayerNorm Class

In the cell below, complete the `forward` method of the `LayerNorm` class. You need to fill in four lines:

1. **Compute the mean** ($\mu$) of `X` along the last dimension
2. **Compute the standard deviation** ($\sigma$) — use `torch.var(..., unbiased=False)`, take the square root, and add $\epsilon$
3. **Normalize** to get $\hat{X} = (X - \mu) / \sigma$
4. **Scale and shift** to get $Y = \gamma \cdot \hat{X} + \beta$

**Useful functions:**
- `X.mean(dim=-1, keepdim=True)` — computes the mean across the last dimension, result shape `(B, T, 1)`
- `X.var(dim=-1, keepdim=True, unbiased=False)` — computes the population variance (divides by $N$, not $N-1$)
- `torch.sqrt(tensor)` — element-wise square root
- Standard arithmetic (`+`, `-`, `*`, `/`) all broadcast automatically when shapes are compatible

**Hints:**
- Use `keepdim=True` so the mean/variance shapes are `(B, T, 1)` and can subtract/divide from `X` of shape `(B, T, C)` via broadcasting
- Compute sigma as `torch.sqrt(variance + self.eps)` — add epsilon *inside* the sqrt for numerical stability
- `self.gamma` and `self.beta` are shape `(C,)` and will broadcast over the `B` and `T` dimensions automatically

### Task 01-1: Code (0.5 pts.)


In [7]:
import torch
import math

from collections import namedtuple

# like a struct in c
class LayerNorm:
    # Named tuple to store intermediate values needed for the backward pass
    Grad_Info = namedtuple('Grad_Info', [
        'x',           # (B, T, C) original input (before LN)
        'x_hat',       # (B, T, C) normalized: (X - mu) / sigma
        'x_norm',      # (B, T, C) after gamma/beta scaling
        'sigma',       # (B, T, 1) std dev (for LN backward)
    ])
    """
    Layer normalization applied over the last dimension.

    For input of shape (B, T, C):
      - B = batch size
      - T = sequence length (number of tokens)
      - C = embedding dimension (this is what we normalize over)

    Attributes:
        output_dim (int): Size of the last dimension to normalize over (= C).
        eps (float): Small constant for numerical stability (default 1e-5).
        gamma (torch.Tensor): Learnable scale parameter, shape (C,), initialized to all 1s.
        beta (torch.Tensor): Learnable shift parameter, shape (C,), initialized to all 0s.
    """
    def __init__(self, output_dim, eps=1e-5, device='cpu'):
        self.output_dim = output_dim
        self.eps = eps

        # gamma=1 and beta=0 means LayerNorm starts as a pure normalizer;
        # during training, the model learns to adjust these per-feature
        self.gamma = torch.ones(output_dim, device=device)
        self.beta  = torch.zeros(output_dim, device=device)

    def forward(self, X):
        """
        Apply layer normalization to the input tensor.

        Args:
            X (torch.Tensor): Input of shape (B, T, C).

        Returns:
            torch.Tensor: Normalized tensor of the same shape as X.
        """
        # Step 1: Compute the mean across the embedding dimension (last dim)
        #   keepdim=True preserves the shape so we can subtract from X via broadcasting
        #   Result shape: (B, T, 1)
        # TODO: Compute the mean of X along the last dimension, keeping dimensions
        # LINES: 1
        mu = X.mean(dim = -1, keepdim = True)

        # Step 2: Compute the standard deviation
        #   We use unbiased=False for the population variance (divide by N, not N-1)
        #   Add eps INSIDE the sqrt for numerical stability
        #   Result shape: (B, T, 1)
        # TODO: Compute the variance of X along the last dimension (unbiased=False), keeping dimensions
        # LINES: 1
        sigma = torch.sqrt(X.var(dim = -1, keepdim = True, unbiased = False) + self.eps)

        # Step 3: Normalize to zero mean and unit variance
        #   This is the x_hat from the formula: (x - mu) / sigma
        #   Result shape: (B, T, C) — same as input
        # TODO: Normalize X to produce X_hat
        # LINES: 1
        X_hat  = (X - mu) / sigma

        # Step 4: Apply learned scale (gamma) and shift (beta)
        #   gamma and beta are both shape (C,) and broadcast over B and T
        #   This lets the model "undo" the normalization if that's useful
        #   Result shape: (B, T, C)
        # TODO: Scale and shift using gamma and beta
        # LINES: 1
        X_norm = self.gamma * X_hat + self.beta

        # Cache intermediate values for the backward pass
        self.cache = LayerNorm.Grad_Info(
            x=X, x_hat=X_hat, x_norm=X_norm, sigma=sigma,
        )

        return X_norm

    def backward(self, dX):
        """Backward pass — computes gradients for gamma, beta, and the input."""
        B, T, C = dX.shape
        D = C   # feature dimension for LayerNorm
        cache = self.cache

        # Gradient through the scale: d(gamma * x_hat + beta)/d(x_hat) = gamma
        dX_hat      = dX * self.gamma

        # Parameter gradients (summed over batch and sequence dimensions)
        self.dgamma = (dX * cache.x_hat).sum(dim=(0, 1))    # (C,)
        self.dbeta  = dX.sum(dim=(0, 1))                     # (C,)

        # Input gradient (the LayerNorm backward formula)
        dX_ln = (1.0 / (D * cache.sigma)) * (
            D * dX_hat
            - dX_hat.sum(dim=-1, keepdim=True)
            - cache.x_hat * (dX_hat * cache.x_hat).sum(dim=-1, keepdim=True)
        )

        return dX_ln

    def update(self, lr):
        """SGD parameter update."""
        self.gamma -= lr * self.dgamma
        self.beta  -= lr * self.dbeta


### Task 01-2: Description (0 pts.)
#### Comparison to `nn.LayerNorm`

PyTorch has a built-in `nn.LayerNorm` that does the same thing we just implemented, but optimized in C++/CUDA. Let's verify our implementation produces the same result!

The test below:
1. Creates a random input tensor of shape `(2, 6, 16)` — 2 batches, 6 tokens, 16-dim embeddings
2. Runs it through **our** LayerNorm and PyTorch's `nn.LayerNorm`
3. Checks that the outputs match (within floating-point tolerance)

### Task 01-2: Code (0.5 pts.)


In [8]:
import torch

torch.manual_seed(42)

B, T, C = 2, 6, 16   # batch=2, sequence length=6, embedding dim=16

X = torch.randn(B, T, C)

# --- Our LayerNorm ---
our_ln = LayerNorm(C)
our_out = our_ln.forward(X)

# --- PyTorch's built-in LayerNorm (same init: gamma=1, beta=0) ---
pyt_ln = torch.nn.LayerNorm(C)
with torch.no_grad():
    # Copy our weights into PyTorch's module so we're comparing apples to apples
    pyt_ln.weight.copy_(our_ln.gamma)
    pyt_ln.bias.copy_(our_ln.beta)
pyt_out = pyt_ln(X)

print("Our LayerNorm output shape:   ", our_out.shape)
print("pytorch LayerNorm output shape:", pyt_out.shape)
print()
print("Our first token embedding (first 8 values):")
print(our_out[0, 0, :8])
print("pytorch first token embedding (first 8 values):")
print(pyt_out[0, 0, :8].detach())
print()
print("Outputs match:", torch.allclose(our_out, pyt_out.detach(), atol=1e-5))


Our LayerNorm output shape:    torch.Size([2, 6, 16])
pytorch LayerNorm output shape: torch.Size([2, 6, 16])

Our first token embedding (first 8 values):
tensor([ 1.7266,  1.3588,  0.8680, -1.6473,  0.6820, -0.9185,  0.0784, -1.2282])
pytorch first token embedding (first 8 values):
tensor([ 1.7266,  1.3588,  0.8680, -1.6473,  0.6820, -0.9185,  0.0784, -1.2282])

Outputs match: True


### Task 01-2: Expected Output (0.5 pts.)
```
Our LayerNorm output shape:    torch.Size([2, 6, 16])
pytorch LayerNorm output shape: torch.Size([2, 6, 16])

Our first token embedding (first 8 values):
tensor([ ...values... ])
pytorch first token embedding (first 8 values):
tensor([ ...values... ])

Outputs match: True
```

**Note:** The exact floating-point values will differ based on your machine, but
`Outputs match: True` is the key result — if you see `False` something is wrong with
your normalization formula.


---

## Task 02: Scaled Dot-Product Self-Attention

### The Big Idea

Self-attention is the mechanism that makes Transformers powerful. The core insight is simple: **when processing a token, the model should be able to "look at" other tokens in the sequence to gather relevant context.**

For example, in the sentence *"The cat sat on the mat because **it** was tired"*, when processing the word "it", the model needs to figure out that "it" refers to "cat" (not "mat"). Self-attention enables this by computing a relevance score between every pair of tokens.

### Queries, Keys, and Values

Self-attention uses three projections of the input, inspired by information retrieval:

- **Query (Q)**: "What am I looking for?" — what this token wants to know
- **Key (K)**: "What do I contain?" — what this token can offer
- **Value (V)**: "What information do I carry?" — the actual content to retrieve

Think of it like a library search:
- You have a **query** (your search terms)
- Each book has **keys** (title, author, subject tags)
- You compute relevance scores (query · key) to find the best matches
- You retrieve the **values** (the actual book contents) of the best matches

### The Attention Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{D_h}}\right) V$$

Let's break this down step by step:

**Step 1 — Compute attention scores:** Multiply $Q$ of shape $(B, T, D_h)$ by $K^\top$ of shape $(B, D_h, T)$ to get a score matrix of shape $(B, T, T)$, where each entry $(i, j)$ is the dot product $q_i \cdot k_j$ measuring how relevant token $j$ is to token $i$.

**Step 2 — Scale and mask:** Divide all scores by $\sqrt{D_h}$ to keep the variance manageable. If using causal masking, set future positions to $-\infty$.

**Step 3 — Softmax and weighted sum:** Apply softmax across the last dimension to get attention weights (each row sums to 1). Multiply the weights $(B, T, T)$ by $V$ $(B, T, D_h)$ to get the output $(B, T, D_h)$ — a weighted combination of value vectors.

**Why scale by $1/\sqrt{D_h}$?** Without scaling, the dot products $QK^\top$ grow proportionally to $D_h$. When these values get large, softmax saturates (outputs are nearly 0 or 1), which makes gradients vanishingly small. Dividing by $\sqrt{D_h}$ keeps the variance of the scores around 1, regardless of dimension size.

### Causal Masking

For language modeling (predicting the next token), we need **causal masking**: token $i$ should only attend to tokens $\leq i$. Without this, the model could "cheat" by looking at future tokens during training.

The mask works by setting all entries above the diagonal in the $(T, T)$ score matrix to $-\infty$ before applying softmax. Since $\text{softmax}(-\infty) = 0$, those future positions receive zero attention weight. The result is a lower-triangular attention pattern: the first token attends only to itself, the second token attends to tokens 1-2, the third to tokens 1-3, and so on.

![Scaled Dot-Product Attention](https://jalammar.github.io/images/t/self-attention-output.png)

*Image: Self-attention computes a weighted sum of value vectors, where the weights come from how well each query matches each key. (Source: [Jay Alammar — The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/))*

### Task 02-1: Description (0 pts.)
#### `scaled_dot_product_attention` Function

Implement the attention formula from above. You need to fill in four parts:

1. **Compute raw scores**: $QK^\top / \sqrt{D_h}$
2. **Apply causal mask** (if `causal=True`): use `torch.tril` + `masked_fill`
3. **Softmax**: convert scores to attention weights (probabilities that sum to 1)
4. **Weighted sum**: multiply attention weights by $V$

**Key functions you'll need:**
- `K.transpose(1, 2)` — transposes the last two dimensions: `(B, T, Dh)` → `(B, Dh, T)`
- `torch.tril(torch.ones(T, T))` — creates a lower-triangular matrix of ones
- `scores.masked_fill(~mask, float('-inf'))` — sets masked positions to $-\infty$
- `F.softmax(scores, dim=-1)` — softmax along the last dimension

### Task 02-1: Code (0.5 pts.)


In [9]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, model_dim, causal=False):
    """
    Compute scaled dot-product attention.

    Args:
        Q (torch.Tensor): Query matrix of shape (B, T, Dh).
        K (torch.Tensor): Key matrix of shape (B, T, Dh).
        V (torch.Tensor): Value matrix of shape (B, T, Dh).
        model_dim (int): Dimension used for scaling (typically Dh).
        causal (bool): If True, apply a causal (lower-triangular) mask so that
                       position i can only attend to positions <= i.

    Returns:
        output (torch.Tensor): Attention-weighted values, shape (B, T, Dh).
        attn   (torch.Tensor): Attention weight matrix, shape (B, T, T).
    """
    B, T, Dh = Q.shape

    # Step 1: Compute raw attention scores
    #   Q @ K^T gives us a (B, T, T) matrix where entry [b, i, j] is
    #   the dot product between token i's query and token j's key.
    #   Divide by sqrt(model_dim) to prevent softmax saturation.
    # TODO: Compute raw attention scores by taking the dot product of Q and K^T,
    #       then scale by 1/sqrt(model_dim).
    # LINES: 1
    scores = (Q @ K.transpose(1, 2)) / math.sqrt(model_dim)

    # Step 2: If causal, build a lower-triangular boolean mask of shape (T, T)
    #   torch.tril creates a matrix where entry [i, j] is True if j <= i
    #   masked_fill sets all False positions to -inf (so softmax gives them weight 0)
    if causal:
        tril = torch.tril(torch.ones(T, T, device=Q.device, dtype=torch.bool))
        scores = scores.masked_fill(~tril, float('-inf'))

    # Step 3: Apply softmax to convert raw scores into attention weights.
    #   After softmax, each row sums to 1 — these are the "attention probabilities"
    #   that determine how much each token attends to every other token.
    # TODO: Apply softmax over the last dimension to get attention weights.
    # LINES: 1
    attn = F.softmax(scores, dim = -1)

    # Step 4: Compute the weighted sum of values.
    #   attn @ V takes the weighted combination of value vectors according
    #   to the attention weights. Each token's output is a blend of all
    #   values it attends to.
    # TODO: Multiply the attention weights by V to produce the output.
    # LINES: 1
    output = attn @ V

    return output, attn


### Task 02-1: Sanity Check (0 pts.)

Run the cell below to verify your implementation. We check:
- Output shape is correct: `(1, 5, 8)`
- Attention weights sum to 1 for each query (they're valid probability distributions)
- With causal masking, the first token can only attend to itself (weight = 1.0)


In [10]:
torch.manual_seed(0)
B, T, model_dim = 1, 5, 8

Q_test = torch.randn(B, T, model_dim)
K_test = torch.randn(B, T, model_dim)
V_test = torch.randn(B, T, model_dim)

# Test without causal masking (all tokens can see all other tokens)
out, attn = scaled_dot_product_attention(Q_test, K_test, V_test, model_dim, causal=False)
print("Output shape:", out.shape)
print("Attention weights sum to 1 (non-causal):", torch.allclose(attn.sum(dim=-1), torch.ones(B, T), atol=1e-5))

# Test WITH causal masking (each token can only see itself and prior tokens)
out_c, attn_c = scaled_dot_product_attention(Q_test, K_test, V_test, model_dim, causal=True)
print("Output shape (causal):", out_c.shape)
print("Attention weights sum to 1 (causal):    ", torch.allclose(attn_c.sum(dim=-1), torch.ones(B, T), atol=1e-5))
print("First token attends only to itself (causal):")
print(attn_c[0, 0])   # should be [1.0, 0.0, 0.0, 0.0, 0.0]


Output shape: torch.Size([1, 5, 8])
Attention weights sum to 1 (non-causal): True
Output shape (causal): torch.Size([1, 5, 8])
Attention weights sum to 1 (causal):     True
First token attends only to itself (causal):
tensor([1., 0., 0., 0., 0.])


### Task 02-1: Expected Output (0.5 pts.)
```
Output shape: torch.Size([1, 5, 8])
Attention weights sum to 1 (non-causal): True
Output shape (causal): torch.Size([1, 5, 8])
Attention weights sum to 1 (causal):     True
First token attends only to itself (causal):
tensor([1., 0., 0., 0., 0.])
```


### Task 02-2: Description (0 pts.)
#### `SingleHeadAttention` Class

The attention function above takes Q, K, V as inputs — but where do they come from? In a Transformer, Q, K, and V are **learned linear projections** of the input:

$$Q = X W_Q^\top + b_Q, \quad K = X W_K^\top + b_K, \quad V = X W_V^\top + b_V$$

The model learns **what to look for** ($W_Q$), **what to advertise** ($W_K$), and **what to send** ($W_V$) entirely through training. After computing attention, there's one more projection:

$$Y = \text{head} \cdot W_O^\top + b_O$$

This output projection $W_O$ lets the model transform the attention output before it gets added back via the residual connection.

The data flow is: the input $X$ of shape $(B, T, C)$ is projected through three separate linear layers to produce $Q$, $K$, and $V$ (each of shape $(B, T, C)$). These are passed into `scaled_dot_product_attention`, which returns the attention output (called `head`) of shape $(B, T, C)$. Finally, `head` is passed through one more linear layer (the output projection) to produce the final output $Y$ of shape $(B, T, C)$.

Complete the `forward` method below. You need to:
1. **Project** X into Q, K, V using the weight matrices and biases
2. **Call** your `scaled_dot_product_attention` function
3. **Apply** the output projection $W_O$

**Useful functions:**
- Matrix multiply in PyTorch: `X @ W.T` computes $X W^\top$ (the `.T` transposes the matrix)
- Adding a bias: `X @ W.T + b` — the bias `b` of shape `(C,)` broadcasts over batch and sequence dims
- Your `scaled_dot_product_attention(Q, K, V, self.model_dim, causal)` returns a tuple `(output, attn_weights)`

### Task 02-2: Code (1 pt.)
![Self-Attention Q/K/V](https://jalammar.github.io/images/t/transformer_self_attention_vectors.png)

*Image: Each input token is projected into Query, Key, and Value vectors through learned weight matrices. (Source: [Jay Alammar — The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/))*


In [11]:
import math
import torch

import torch.nn.functional as F
from collections import namedtuple

class SelfAttention:
    # Cache for intermediate values needed in the backward pass
    Grad_Info = namedtuple('Grad_Info', [
        'x',                               # (B, T, C) original input
        'q', 'k', 'v',                     # (B, T, Dh) projected queries, keys, values
        'scores',                           # (B, T, T)  raw attention scores
        'attn',                             # (B, T, T)  attention weights (after softmax)
        'attn_v',                           # (B, T, Dh) attention output (before Wo)
    ])

    def __init__(self, model_dim, device='cpu'):
        self.model_dim = model_dim
        self.device = device

        # Initialize Q, K, V, O projection matrices with Xavier scaling.
        # Xavier init keeps the variance of activations stable across layers.
        scale_in = math.sqrt(1.0 / self.model_dim)
        self.Wq = torch.randn(self.model_dim, self.model_dim, device=self.device) * scale_in
        self.Wk = torch.randn(self.model_dim, self.model_dim, device=self.device) * scale_in
        self.Wv = torch.randn(self.model_dim, self.model_dim, device=self.device) * scale_in
        self.bq = torch.zeros(self.model_dim, device=self.device)
        self.bk = torch.zeros(self.model_dim, device=self.device)
        self.bv = torch.zeros(self.model_dim, device=self.device)

        # Output projection: maps attention output back to model_dim
        self.Wo = torch.randn(self.model_dim, self.model_dim, device=self.device) * scale_in
        self.bo = torch.zeros(self.model_dim, device=self.device)

        self.cache = None

    def forward(self, X, causal=False):
        """
        Forward pass of single-head self-attention.

        Args:
            X (torch.Tensor): Input of shape (B, T, C).
            causal (bool): Whether to apply causal masking.

        Returns:
            y_ctx (torch.Tensor): Output of shape (B, T, C).
            attn  (torch.Tensor): Attention weights of shape (B, T, T).
        """
        B, T, C = X.shape

        # Step 1: Project input X into Queries, Keys, and Values.
        #   Each projection is a linear transformation: X @ W^T + b
        #   This is the same operation as our LinearLayer, just done manually.
        # TODO: Project X into Q, K, and V using the weight matrices and biases.
        # LINES: 3
        Q = X @ self.Wq.T + self.bq
        K = X @ self.Wk.T + self.bk
        V = X @ self.Wv.T + self.bv

        # Step 2: Compute attention using our scaled_dot_product_attention function.
        #   This returns both the output (weighted sum of V) and the attention weights.
        # TODO: Call your scaled_dot_product_attention function to get head and attn.
        # LINES: 1
        head, attn = scaled_dot_product_attention(Q, K, V, self.model_dim, causal)

        # Step 3: Apply the output projection to map back to model_dim.
        #   This final linear layer lets the model transform the attention output
        #   before it gets added to the residual stream.
        # TODO: Apply the output projection Wo and bias bo to map back to model_dim.
        # LINES: 1
        y_ctx = head @ self.Wo.T + self.bo

        # Cache values for backward pass
        self.cache = SelfAttention.Grad_Info(
            x=X, q=Q, k=K, v=V, scores=attn, attn=attn, attn_v=head
        )

        return y_ctx, attn

    def backward(self, dY):
        """Backward pass — computes gradients for all projection matrices."""
        B, T, C = dY.shape
        dy_ctx = dY

        # Output projection backward
        self.dWo = dy_ctx.reshape(-1, C).T @ self.cache.attn_v.reshape(-1, self.model_dim)
        self.dbo = dy_ctx.sum(dim=(0, 1))
        dattn_v  = dy_ctx @ self.Wo

        # Attention backward
        dv    = self.cache.attn.transpose(1, 2) @ dattn_v
        dattn = dattn_v @ self.cache.v.transpose(1, 2)

        # Softmax backward
        tmp     = (dattn * self.cache.attn).sum(dim=-1, keepdim=True)
        dscores = (dattn - tmp) * self.cache.attn

        factor = 1.0 / math.sqrt(self.model_dim)
        dqk = dscores * factor

        dq = dqk @ self.cache.k
        dk = dqk.transpose(1, 2) @ self.cache.q

        # Q/K/V parameter gradients
        self.dWq = dq.reshape(-1, self.model_dim).T @ self.cache.x.reshape(-1, self.model_dim)
        self.dbq = dq.sum(dim=(0, 1))
        self.dWk = dk.reshape(-1, self.model_dim).T @ self.cache.x.reshape(-1, self.model_dim)
        self.dbk = dk.sum(dim=(0, 1))
        self.dWv = dv.reshape(-1, self.model_dim).T @ self.cache.x.reshape(-1, self.model_dim)
        self.dbv = dv.sum(dim=(0, 1))

        # Gradient w.r.t. input X (sum of all three branches)
        dX = dq @ self.Wq + dk @ self.Wk + dv @ self.Wv
        return dX

    def update(self, lr):
        """SGD parameter update for all projection matrices."""
        self.Wq -= lr * self.dWq; self.bq -= lr * self.dbq
        self.Wk -= lr * self.dWk; self.bk -= lr * self.dbk
        self.Wv -= lr * self.dWv; self.bv -= lr * self.dbv
        self.Wo -= lr * self.dWo; self.bo -= lr * self.dbo


### Task 02-3: Description (0 pts.)
#### Comparison to `F.scaled_dot_product_attention`

PyTorch 2.0+ ships a fused `F.scaled_dot_product_attention` kernel that's heavily optimized (it uses Flash Attention under the hood for GPU efficiency). Let's verify our implementation produces the same result by sharing the *same* $W_Q, W_K, W_V, W_O$ weights between our class and a manual PyTorch reference.

### Task 02-3: Code (0.5 pts.)


In [12]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)

B, T, C = 1, 6, 16

X_test = torch.randn(B, T, C)

# Create our SelfAttention with random weights
sha = SelfAttention(model_dim=C)

# ---- Our output ----
our_Y, _ = sha.forward(X_test)

# ---- PyTorch reference (using the SAME weights from our class) ----
with torch.no_grad():
    # Step 1: Project X using our learned weights
    Q_ref = X_test @ sha.Wq.T + sha.bq
    K_ref = X_test @ sha.Wk.T + sha.bk
    V_ref = X_test @ sha.Wv.T + sha.bv

    # Step 2: PyTorch's fused attention (no causal mask, same scaling)
    head_ref = F.scaled_dot_product_attention(Q_ref, K_ref, V_ref, is_causal=False)

    # Step 3: Output projection
    Y_ref = head_ref @ sha.Wo.T + sha.bo

print("Our output shape:       ", our_Y.shape)
print("Reference output shape: ", Y_ref.shape)
print()
print("Our first token (first 8 values):")
print(our_Y[0, 0, :8])
print("Reference first token (first 8 values):")
print(Y_ref[0, 0, :8])
print()
print("Outputs match:", torch.allclose(our_Y, Y_ref, atol=1e-5))


Our output shape:        torch.Size([1, 6, 16])
Reference output shape:  torch.Size([1, 6, 16])

Our first token (first 8 values):
tensor([ 1.9029,  0.7499,  0.2943, -0.2866,  0.5785, -0.9556, -0.7967,  1.4096])
Reference first token (first 8 values):
tensor([ 1.9029,  0.7499,  0.2943, -0.2866,  0.5785, -0.9556, -0.7967,  1.4096])

Outputs match: True


### Task 02-3: Expected Output (0.5 pts.)
```
Our output shape:        torch.Size([1, 6, 16])
Reference output shape:  torch.Size([1, 6, 16])

Our first token (first 8 values):
tensor([ ...values... ])
Reference first token (first 8 values):
tensor([ ...values... ])

Outputs match: True
```


---

## Task 03: Feed-Forward Sub-Block

### What is the FFN?

After self-attention gathers context from other tokens, the **feed-forward network (FFN)** processes each token *independently* through a small two-layer neural network. Think of it as the "thinking" step: attention says "here's the relevant context," and the FFN says "now let me process this information."

$$\text{FFN}(X) = \text{ReLU}(X W_1^\top + b_1)\, W_2^\top + b_2$$

### The Expand-and-Contract Pattern

The FFN expands the dimension by a factor of 4, applies ReLU, then contracts back. Starting from the input of shape $(B, T, C)$ (e.g., $(2, 8, 32)$), the first linear layer $W_1$ maps from $C$ to $4C$, expanding each token to a 128-dimensional hidden representation. ReLU zeros out any negative values. Then the second linear layer $W_2$ maps from $4C$ back to $C$, compressing each token back to its original 32-dimensional size.

**Why expand by 4x?** The expansion gives the network more "room to think" in a higher-dimensional space, where it can compute more complex features. The contraction then compresses this back to the original size so it can be added via the residual connection. The 4x factor is a convention from the original Transformer paper.

**Why process tokens independently?** Self-attention already handled token-to-token interaction. The FFN adds per-token nonlinear processing — similar to how a standard feedforward network works, but applied independently to every position in the sequence.

### Task 03-1: Description (0 pts.)
#### Combined: FFN + TransformerBlock

We now have all the pieces. A **Pre-LayerNorm Transformer Block** chains them as follows:

**Sub-block 1 (Attention):** The input $X$ is passed through Layer Norm 1, then through Single-Head Self-Attention, and the result is added back to the original $X$ (residual connection). This produces an intermediate result $Y_\text{attn}$.

**Sub-block 2 (FFN):** $Y_\text{attn}$ is passed through Layer Norm 2, then through the FFN (Linear, ReLU, Linear), and the result is added back to $Y_\text{attn}$ (another residual connection). This produces the final output $Y$.

Both sub-blocks follow the same pattern: **normalize, transform, add residual.**

**Key design details:**

- **Pre-LayerNorm** means we normalize *before* each sub-block (attention and FFN), not after. This is the modern convention (used by GPT-2, GPT-3, LLaMA, etc.) because it produces more stable training.

- **Residual connections** (the `+` additions) add the sub-block's input back to its output. This gives the model a "highway" to pass information through unchanged. Without residuals, deep transformers (12+ layers) are nearly impossible to train because gradients vanish.

In the cell below, the FFN is built inline using `LinearLayer` and `ReLU` from Lab 02. Complete the `forward` method of `TransformerBlock` — you need to:

1. **Sub-block 1**: Apply LN1 → Self-Attention → add residual
2. **Sub-block 2**: Apply LN2 → FFN (Linear → ReLU → Linear) → add residual

**Useful functions:**
- `self.LN1.forward(X)` — apply Layer Norm 1
- `self.SA.forward(X, causal)` — returns `(output, attn_weights)` — you only need the output, so use `X, _ = self.SA.forward(...)`
- `self.LN2.forward(X)` — apply Layer Norm 2
- `self.L1.forward(X)` — first FFN linear layer (expands $C \to 4C$)
- `self.R.forward(X)` — ReLU activation
- `self.L2.forward(X)` — second FFN linear layer (contracts $4C \to C$)
- The residual pattern is: save the input, transform it, then add the saved input back with `+=`

### Task 03-1: Code (0.5 pts.)
![Transformer Block](https://jalammar.github.io/images/xlnet/transformer-encoder-block-2.png)

*Image: The Transformer block with LayerNorm, Self-Attention, and FFN sub-blocks connected by residual connections. (Source: [Jay Alammar](https://jalammar.github.io/illustrated-transformer/))*


In [13]:
class TransformerBlock:
    """
    A single Pre-LayerNorm Transformer block:
    X → (LN1 → SingleHeadAttention → Residual) → (LN2 → FFN → Residual) → Y

    Attributes:
        LN1  (LayerNorm): Layer norm before attention.
        SA   (SelfAttention): Single-head self-attention.
        LN2  (LayerNorm): Layer norm before FFN.
        L1   (LinearLayer): FFN first linear layer (C → 4C expansion).
        R    (ReLU): Activation function.
        L2   (LinearLayer): FFN second linear layer (4C → C contraction).
    """
    def __init__(self, model_dim, ffn_expand=4, device='cpu'):
        self.device = device

        # FFN hidden dimension = 4x the model dimension
        ffn_dim = ffn_expand * model_dim

        # Sub-block 1 components: LayerNorm + Self-Attention
        self.LN1 = LayerNorm(model_dim, device=device)
        self.SA = SelfAttention(model_dim, device=device)

        # Sub-block 2 components: LayerNorm + FFN (two linear layers with ReLU)
        self.LN2 = LayerNorm(model_dim, device=device)
        self.L1 = LinearLayer(model_dim, ffn_dim, device)   # expand:  C → 4C
        self.R = ReLU()                                      # activation
        self.L2 = LinearLayer(ffn_dim, model_dim, device)   # contract: 4C → C

    def forward(self, X, causal=False):
        """
        Run the forward pass of the transformer block.

        Args:
            X      (torch.Tensor): Input embeddings of shape (B, T, C).
            causal (bool): If True, apply causal masking in self-attention.

        Returns:
            Y (torch.Tensor): Block output of shape (B, T, C).
        """

        # ── Sub-block 1: LayerNorm → Self-Attention → Residual ────────────
        #
        # Save X as the residual, normalize, run attention, add residual back.
        # The residual connection means the attention output is ADDED to the
        # original input, not replacing it.

        # TODO: Apply LN1 to X, then pass through self-attention (capture only the
        #       output, not the attention weights), then add the residual X.
        # LINES: 4
        resid = X

        X = self.LN1.forward(X)
        X, _ = self.SA.forward(X)

        X += resid

        # ── Sub-block 2: LayerNorm → FFN → Residual ─────────────────────
        #
        # Same pattern: save residual, normalize, run FFN
        # (Linear1 → ReLU → Linear2), add residual back.

        # TODO: Apply LN2, then pass through FFN (L1 → R → L2),
        #       then add the residual.
        # LINES: 6
        resid = X

        X = self.LN2.forward(X)
        X = self.L1.forward(X)
        X = self.R.forward(X)
        X = self.L2.forward(X)

        Y = X + resid

        return Y

    def backward(self, dY):
        """Backward pass through the full transformer block (reverse order)."""
        d_resid = dY

        # FFN backward (reverse order: L2 → ReLU → L1 → LN2)
        dY = self.L2.backward(dY)
        dY = self.R.backward(dY)
        dY = self.L1.backward(dY)
        dY = self.LN2.backward(dY)

        dY += d_resid  # residual gradient

        d_resid = dY

        # Attention backward (reverse order: SA → LN1)
        dY = self.SA.backward(dY)
        dY = self.LN1.backward(dY)

        return dY + d_resid  # residual gradient

    def update(self, lr):
        """SGD update for all sub-modules."""
        self.LN1.update(lr)
        self.SA.update(lr)
        self.LN2.update(lr)
        self.L1.update(lr)
        self.R.update(lr)
        self.L2.update(lr)

### Task 03-1: Sanity Check (0 pts.)

The key thing to verify: the output shape must match the input shape. A Transformer block is a **shape-preserving** operation — this is what allows us to stack multiple blocks on top of each other.


In [14]:
torch.manual_seed(42)
B, T, C = 2, 8, 32

X_blk = torch.randn(B, T, C)

block = TransformerBlock(model_dim=C)
Y_blk = block.forward(X_blk, causal=False)

print("Block input  shape:", X_blk.shape)
print("Block output shape:", Y_blk.shape)
print("Shapes match:", Y_blk.shape == X_blk.shape)


Block input  shape: torch.Size([2, 8, 32])
Block output shape: torch.Size([2, 8, 32])
Shapes match: True


### Task 03-1: Expected Output (0.5 pts.)
```
Block input  shape: torch.Size([2, 8, 32])
Block output shape: torch.Size([2, 8, 32])
Shapes match: True
```


---

## Task 04: Causal Masking Demo

### Why Causal Masking Matters

When training a language model, we feed in a sequence like `"The cat sat on"` and ask the model to predict the next token at every position:
- Position 0 (`"The"`) should predict `"cat"` — using only `"The"`
- Position 1 (`"cat"`) should predict `"sat"` — using `"The"` and `"cat"`
- Position 2 (`"sat"`) should predict `"on"` — using `"The"`, `"cat"`, and `"sat"`

Without causal masking, position 0 could "cheat" by looking at position 1 to see that `"cat"` comes next. Causal masking prevents this by zeroing out attention to future positions.

### What to Look For

The cell below encodes the prompt `"Thou art"` and runs it through a Transformer block with and without causal masking. Compare the attention weight vectors for the **last token** — without masking, it attends to all positions; with masking, it only attends to positions $\leq$ its own index.

### Task 04-1: Code (0 pts.) — just run and observe


In [15]:
import torch
import irishGPT.tokenizer as tokenizer

torch.manual_seed(42)

# Load the Shakespeare tokenizer from the course repo
r_t = tokenizer.Regex_Tokenizer()
r_t.load('Datasets/shakespeare_vocab.json')

# Encode a short prompt into token IDs
prompt = 'Thou art'
token_ids = r_t.encode('<|sos|>' + prompt + '<|eos|>')[:-1]  # drop final eos
print('Prompt:', repr(prompt))
print('Token ids:', token_ids)

T_len = len(token_ids)
C     = 32

# Build a small random embedding table and embed the prompt
torch.manual_seed(0)
embed_W = torch.randn(len(r_t.vocab), C)                       # (vocab_size, C)
tokens  = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)  # (1, T)
X_demo  = embed_W[tokens]   # (1, T, C)  — look up each token's embedding

# Run through a Transformer block
block_demo = TransformerBlock(model_dim=C)

# Get attention weights with and without causal masking
# (We call SA directly after LN1 to inspect the raw attention weights)
_, attn_non_causal = block_demo.SA.forward(block_demo.LN1.forward(X_demo), causal=False)
_, attn_causal     = block_demo.SA.forward(block_demo.LN1.forward(X_demo), causal=True)

# Round for readability
def fmt(t): return (t * 100).round() / 100

print()
print("=== Attention weights for the LAST token ===")
print(f"{'Position':<12}", [f't{i}' for i in range(T_len)])
print(f"{'Non-causal':<12}", fmt(attn_non_causal[0, -1]).tolist())
print(f"{'Causal':<12}    ", fmt(attn_causal[0, -1]).tolist())
print()
print("Non-causal: last token can attend to all positions (weights spread across all)")
print("Causal:     last token attends only to positions <= its own index")


Prompt: 'Thou art'
Token ids: [257, 84, 104, 262, 261, 114, 116]

=== Attention weights for the LAST token ===
Position     ['t0', 't1', 't2', 't3', 't4', 't5', 't6']
Non-causal   [0.07000000029802322, 0.4399999976158142, 0.009999999776482582, 0.20000000298023224, 0.07999999821186066, 0.20000000298023224, 0.009999999776482582]
Causal           [0.07000000029802322, 0.4399999976158142, 0.009999999776482582, 0.20000000298023224, 0.07999999821186066, 0.20000000298023224, 0.009999999776482582]

Non-causal: last token can attend to all positions (weights spread across all)
Causal:     last token attends only to positions <= its own index


### Task 04-2: Short Answer (0.5 pts.)

After running the cell above, answer the following:

**Question:** Look at the causal attention weights for the *first* token (`t0`).
It can only attend to itself. Does that mean the transformer block output for
`t0` is the same whether we use causal masking or not? Why or why not?

**Hint:** Think about what happens after attention — does the FFN sub-block depend on other tokens?

**Answer:** No, the transformer block output for t0 is not the same in the two cases.
Under causal masking, t0 can only attend to itself, so its attention output is essentially just its own value vector v_0.
Under non-causal attention, t0's query attends to all tokens in the sequence, so its attention output is a weighted blend of every value vector v. Information from future tokens gets mixed into t0's representation.


---

## Task 05: Generate Submission

Once you've completed all tasks and verified your outputs, uncomment the function call below and run it. This will convert your notebook to HTML and download it for submission on Canvas.

### Task 05: Code (0 pts.)


In [16]:
import os, json

def export_notebook():
    L_PATH = "nd-cse-10124-lectures/Notebooks"
    L = "Lab_03_Transformer_Block"

    try:
        from google.colab import _message, files

        repo_ipynb_path = f"/content/{L_PATH}/{L}.ipynb"
        nb = _message.blocking_request("get_ipynb", timeout_sec=1)["ipynb"]

        os.makedirs(os.path.dirname(repo_ipynb_path), exist_ok=True)
        with open(repo_ipynb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f)

        !jupyter nbconvert --to html "{repo_ipynb_path}"
        files.download(repo_ipynb_path.replace(".ipynb", ".html"))
    except:
        import subprocess
        nb_fp = os.getcwd() + f'/{L}.ipynb'
        subprocess.run(["jupyter", "nbconvert", "--to", "html", nb_fp], check=True)

# TODO: Uncomment the line below and run to generate the file to submit on canvas
export_notebook()


[NbConvertApp] Converting notebook /content/nd-cse-10124-lectures/Notebooks/Lab_03_Transformer_Block.ipynb to html
[NbConvertApp] Writing 415982 bytes to /content/nd-cse-10124-lectures/Notebooks/Lab_03_Transformer_Block.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>